> - ####📊 Pharma Claims Data Pipeline (Databricks)
 
End-to-end pharma claims pipeline built using a layered architecture in Databricks SQL.
Processes raw claims data through cleansing, deduplication, and validation steps to ensure data quality.
Integrates MDM reference data using ICD-10 (disease) and NDC (drug) standards.
Applies business logic to compute revenue and analytical metrics.
Delivers KPI-driven reporting including claim volume, revenue, and top-performing drugs.
Demonstrates a production-style data pipeline for healthcare analytics.

#####CATALOG + SCHEMA SETUP

In [0]:
%sql
-- Set catalog
USE CATALOG pharma;

-- Create schemas (layers)
CREATE SCHEMA IF NOT EXISTS pharma.ref;
CREATE SCHEMA IF NOT EXISTS pharma.bronze;
CREATE SCHEMA IF NOT EXISTS pharma.silver;
CREATE SCHEMA IF NOT EXISTS pharma.gold;

#####MDM LAYER (REFERENCE DATA)

In [0]:
%sql
-- Drug MDM (50 rows)
CREATE OR REPLACE TABLE pharma.ref.ref_drug_mdm AS
SELECT *
FROM VALUES
('D001','Keytruda','00006-3026-02',1,1),
('D002','Opdivo','00003-3772-11',1,1),
('D003','Revlimid','59572-501-21',1,1),
('D004','Imbruvica','57962-140-12',1,1),
('D005','Tagrisso','0310-0750-30',1,1),
('D006','Darzalex','57894-050-01',1,1),
('D007','Tecentriq','50242-917-01',1,1),
('D008','Rituxan','50242-051-21',1,1),
('D009','Herceptin','50242-134-01',1,1),
('D010','Avastin','50242-060-01',1,1),

('D011','Humira','0074-4339-02',1,0),
('D012','Enbrel','58406-0435-01',1,0),
('D013','Stelara','57894-060-01',1,0),
('D014','Cosentyx','0078-0911-61',1,0),
('D015','Dupixent','00024-5914-01',1,0),

('D016','Ibrance','00069-0419-30',1,1),
('D017','Xtandi','0469-0600-30',1,1),
('D018','Lynparza','0310-0650-30',1,1),
('D019','Verzenio','00002-4471-54',1,1),
('D020','Kisqali','0078-0931-61',1,1),

('D021','Metformin','00378-1010-01',1,0),
('D022','Atorvastatin','00093-7424-56',1,0),
('D023','Amlodipine','00093-7371-56',1,0),
('D024','Losartan','00093-7211-56',1,0),
('D025','Omeprazole','00093-7425-56',1,0),

('D026','Albuterol','00085-1132-01',1,0),
('D027','Levothyroxine','00093-7245-56',1,0),
('D028','Gabapentin','00093-7423-56',1,0),
('D029','Sertraline','00093-7431-56',1,0),
('D030','Fluoxetine','00093-7422-56',1,0),

('D031','Insulin Glargine','00002-8215-01',1,0),
('D032','Insulin Lispro','00002-7510-01',1,0),
('D033','Warfarin','00093-7200-56',1,0),
('D034','Clopidogrel','00093-7234-56',1,0),
('D035','Aspirin','00904-1234-01',1,0),

('D036','Prednisone','00054-0450-25',1,0),
('D037','Amoxicillin','00093-4175-56',1,0),
('D038','Azithromycin','00093-4165-56',1,0),
('D039','Ciprofloxacin','00093-4170-56',1,0),
('D040','Doxycycline','00093-4180-56',1,0),

('D041','Morphine','00054-0234-25',1,0),
('D042','Oxycodone','00054-0456-25',1,0),
('D043','Fentanyl','00054-1234-25',1,0),
('D044','Hydrocodone','00054-5678-25',1,0),
('D045','Tramadol','00093-7426-56',1,0),

('D046','Vitamin D','00093-9001-56',1,0),
('D047','Calcium','00093-9002-56',1,0),
('D048','Iron','00093-9003-56',1,0),
('D049','Zinc','00093-9004-56',1,0),
('D050','Magnesium','00093-9005-56',1,0)
AS t(drug_code, drug_name, ndc_code, active_flag, pdrp_flag);

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ICD MDM (50 rows)
CREATE OR REPLACE TABLE pharma.ref.ref_icd_mdm AS
SELECT *
FROM VALUES
('C50.9','Breast Cancer',1),('C34.90','Lung Cancer',1),('C18.9','Colon Cancer',1),
('C61','Prostate Cancer',1),('C25.9','Pancreatic Cancer',1),
('C71.9','Brain Tumor',1),('C64.9','Kidney Cancer',1),
('C67.9','Bladder Cancer',1),('C22.9','Liver Cancer',1),
('C15.9','Esophageal Cancer',1),

('E11.9','Diabetes',0),('I10','Hypertension',0),('J45.909','Asthma',0),
('E78.5','Hyperlipidemia',0),('M54.5','Back Pain',0),
('F41.9','Anxiety',0),('F32.9','Depression',0),
('K21.9','GERD',0),('N18.9','CKD',0),
('I25.10','CAD',0),

('Z00.00','General Checkup',0),('Z23','Vaccination',0),
('Z79.899','Drug Therapy',0),('Z12.11','Screening',0),
('Z01.818','Pre-op',0)
AS t(icd_code, disease_name, oncology_flag);

num_affected_rows,num_inserted_rows


#####RAW DATA - Way to create sample data

In [0]:
%sql
CREATE OR REPLACE TABLE pharma.bronze.bronze_claims_raw AS
SELECT
    CONCAT('P', id) AS patient_id,
    CONCAT('C', id % 1200) AS claim_id,

    CASE 
        WHEN id % 30 = 0 THEN NULL
        ELSE DATE_ADD(DATE '2023-01-01', CAST(id % 365 AS INT))
    END AS claim_date,

    CONCAT('D', LPAD(CAST((id % 50)+1 AS STRING),3,'0')) AS drug_code,

    element_at(
        array('C50.9','C34.90','C18.9','C61','C25.9','E11.9','I10'),
        CAST((id % 7)+1 AS INT)
    ) AS icd_code,

    CASE WHEN id % 20 = 0 THEN NULL ELSE (id % 5)+1 END AS quantity,

    CASE 
        WHEN id % 50 < 20 THEN (id % 10000) + 5000
        ELSE (id % 500) + 100
    END AS price,

    CONCAT('DR', id % 200) AS doctor_id,
    CONCAT('H', id % 50) AS hospital_id,

    CASE WHEN id % 7 = 0 THEN 'REJECTED' ELSE 'PAID' END AS payment_status

FROM RANGE(1500);

num_affected_rows,num_inserted_rows


#####Staging Layer - Removing Nulls

In [0]:
%sql
-- Clean + validate
CREATE OR REPLACE TABLE pharma.silver.stg_claims AS
SELECT *
FROM pharma.bronze.bronze_claims_raw
WHERE claim_date IS NOT NULL
  AND quantity IS NOT NULL;

num_affected_rows,num_inserted_rows


#####FACT Layer - Removing Dedup

In [0]:
%sql
CREATE OR REPLACE TABLE pharma.silver.fact_claims AS
WITH dedup AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY claim_date DESC) rn
    FROM pharma.silver.stg_claims
)
SELECT * FROM dedup WHERE rn = 1;

num_affected_rows,num_inserted_rows


#####MDM Layer - Joining data with Master Data Layer

In [0]:
%sql
CREATE OR REPLACE TABLE pharma.silver.fact_enriched AS
SELECT
    f.*,
    d.drug_name,
    d.ndc_code,
    d.active_flag,
    d.pdrp_flag,
    i.disease_name,
    i.oncology_flag
FROM pharma.silver.fact_claims f
LEFT JOIN pharma.ref.ref_drug_mdm d
ON f.drug_code = d.drug_code
LEFT JOIN pharma.ref.ref_icd_mdm i
ON f.icd_code = i.icd_code;

num_affected_rows,num_inserted_rows


#####Intermediate Layer - Transforming Data

In [0]:
%sql
CREATE OR REPLACE TABLE pharma.silver.int_claims AS
WITH base AS (
    SELECT *,
        quantity * price AS revenue
    FROM pharma.silver.fact_enriched
    WHERE active_flag = 1
      AND payment_status = 'PAID'
),
calc AS (
    SELECT *,
        SUM(revenue) OVER () AS total_revenue,
        SUM(revenue) OVER (PARTITION BY drug_name) AS drug_revenue
    FROM base
)
SELECT * FROM calc;

num_affected_rows,num_inserted_rows


#####Reporting Layer - KPI

In [0]:
%sql
CREATE OR REPLACE TABLE pharma.gold.rpt_claims_kpi AS
WITH agg AS (
    SELECT
        drug_name,
        COUNT(*) AS total_claims,
        SUM(revenue) AS total_revenue,
        AVG(revenue) AS avg_revenue
    FROM pharma.silver.int_claims
    GROUP BY drug_name
),
ranked AS (
    SELECT *,
        SUM(total_revenue) OVER () AS overall_revenue,
        RANK() OVER (ORDER BY total_revenue DESC) AS rank
    FROM agg
)
SELECT
    drug_name,
    total_claims,
    total_revenue,
    avg_revenue,
    ROUND((total_revenue/overall_revenue)*100,2) AS contribution_pct,
    rank
FROM ranked;

num_affected_rows,num_inserted_rows


#####Quality Checks
- Row count validation
- Referential integrity check
- Business validation

In [0]:
%sql
-- Row count validation
SELECT
 (SELECT COUNT(*) FROM pharma.bronze.bronze_claims_raw) AS bronze,
 (SELECT COUNT(*) FROM pharma.silver.fact_claims) AS fact;

-- Referential integrity check
SELECT *
FROM pharma.silver.fact_enriched
WHERE drug_name IS NULL 
   OR disease_name IS NULL;

-- Business validation (only PAID records expected in final layer)
SELECT *
FROM pharma.silver.int_claims
WHERE payment_status != 'PAID';

patient_id,claim_id,claim_date,drug_code,icd_code,quantity,price,doctor_id,hospital_id,payment_status,rn,drug_name,ndc_code,active_flag,pdrp_flag,disease_name,oncology_flag,revenue,total_revenue,drug_revenue


#####Total Claims
- Volume of processed claims
- Helps track demand / activity level

In [0]:
%sql
SELECT COUNT(*) FROM pharma.silver.int_claims;

COUNT(*)
960


#####Total Revenue
- Overall business revenue

In [0]:
%sql
SELECT SUM(revenue) FROM pharma.silver.int_claims;

SUM(revenue)
7789213


#####Average Revenue per Claim
- Efficiency / value per claim

In [0]:
%sql
SELECT AVG(revenue) FROM pharma.silver.int_claims;

AVG(revenue)
8113.763541666666


#####Oncology vs Non-Oncology Revenue

In [0]:
%sql
SELECT 
    oncology_flag,
    SUM(revenue) AS total_revenue
FROM pharma.silver.int_claims
GROUP BY oncology_flag;

oncology_flag,total_revenue
1,5264248
0,2524965


#####Paid vs Rejected Claims %

In [0]:
%sql
SELECT 
    payment_status,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() AS pct
FROM pharma.silver.fact_claims
GROUP BY payment_status;

payment_status,pct
PAID,85.71428571428571
REJECTED,14.28571428571429


#####Average Drug Price Trend

In [0]:
%sql
SELECT 
    drug_name,
    Round(AVG(price),2) AS avg_price
FROM pharma.silver.int_claims
GROUP BY drug_name;

drug_name,avg_price
Albuterol,375.0
Amlodipine,372.0
Amoxicillin,386.0
Aspirin,374.48
Atorvastatin,361.0
Avastin,5873.29
Azithromycin,372.0
Calcium,396.0
Ciprofloxacin,380.86
Clopidogrel,366.33


#####Top 5 Hospitals by Revenue

In [0]:
%sql
SELECT 
    hospital_id,
    SUM(revenue) AS revenue
FROM pharma.silver.int_claims
GROUP BY hospital_id
ORDER BY revenue DESC
LIMIT 5;

hospital_id,revenue
H4,617670
H9,616695
H19,613995
H14,582900
H3,493452


#####Patient-Level Spend (High Value Patients)

In [0]:
%sql
SELECT 
    patient_id,
    SUM(revenue) AS total_spend
FROM pharma.silver.int_claims
GROUP BY patient_id
ORDER BY total_spend DESC
LIMIT 10;

patient_id,total_spend
P1459,32295
P1454,32270
P1419,32095
P1409,32045
P1404,32020
P1369,31845
P1364,31820
P1359,31795
P1354,31770
P1319,31595


#####Claim Trend Over Time

In [0]:
%sql
SELECT 
    DATE_TRUNC('month', claim_date) AS month,
    COUNT(*) AS claims,
    SUM(revenue) AS revenue
FROM pharma.silver.int_claims
GROUP BY month
ORDER BY month;

month,claims,revenue
2023-01-01T00:00:00.000Z,74,606833
2023-02-01T00:00:00.000Z,69,532683
2023-03-01T00:00:00.000Z,72,664015
2023-04-01T00:00:00.000Z,73,489177
2023-05-01T00:00:00.000Z,73,612375
2023-06-01T00:00:00.000Z,73,569297
2023-07-01T00:00:00.000Z,73,586266
2023-08-01T00:00:00.000Z,76,656000
2023-09-01T00:00:00.000Z,80,674062
2023-10-01T00:00:00.000Z,102,728351


#####Data Loss % (Raw → Final)

In [0]:
%sql
SELECT
    (1 - final_cnt * 1.0 / raw_cnt) * 100 AS data_loss_pct
FROM (
    SELECT 
        (SELECT COUNT(*) FROM pharma.bronze.bronze_claims_raw) raw_cnt,
        (SELECT COUNT(*) FROM pharma.silver.int_claims) final_cnt
);

data_loss_pct
36.00000000000


#####Duplicate Rate

In [0]:
%sql
SELECT 
    COUNT(*) - COUNT(DISTINCT claim_id) AS duplicate_count
FROM pharma.bronze.bronze_claims_raw;

duplicate_count
300
